# Validação — `state_of_data_gold` (v1) vs `state_of_data_gold_v2` (v2)

**Notebook de validação apenas — não faz parte da entrega final do Tech Challenge.**

Objetivo: comparar as 8 tabelas Gold entre o database antigo
(`state_of_data_gold`, bucket anterior à separação do pipeline) e o novo
(`state_of_data_gold_v2`, bucket criado depois da separação em 3 Glue Jobs),
conferindo:

- quantidade de linhas por tabela;
- linhas presentes em um lado e ausentes no outro;
- valores de `count` divergentes para a mesma combinação de dimensões.

Nomes de database/tabela vieram de `../sql/criar_tabelas_gold_athena.sql`
(mesmas 8 tabelas, mesmo schema, só o bucket/`LOCATION` muda entre v1 e v2);
as perguntas de negócio que essas tabelas respondem estão em
`../sql/queries_athena.sql`.

Rode as células em ordem, de cima pra baixo.

In [ ]:
# Instala a lib usada pra consultar o Athena e já devolver pandas.DataFrame
# (não está no seu ambiente hoje). Só precisa rodar uma vez; se o kernel não
# enxergar o import logo depois de instalar, reinicie o kernel e rode nas
# células seguintes de novo.
%pip install -q awswrangler
# %pip install -q boto3

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 3.9.0 requires botocore<1.43.57,>=1.43.3, but you have botocore 1.43.89 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


## 1. Credenciais do AWS Academy Lab

Cole abaixo as credenciais temporárias do **AWS Details** do seu Lab. Elas
expiram a cada sessão — se aparecer erro de autenticação/token expirado,
volte aqui, cole as credenciais atualizadas e rode as células de novo.

In [ ]:
AWS_CREDENTIALS = {
    "aws_access_key_id": "",       # AWSAccessKeyId
    "aws_secret_access_key": "",   # AWSSecretKey
    "aws_session_token": "",       # AWSSessionToken (obrigatório em credenciais temporárias do Lab)
    "region_name": "us-east-1",    # região do Lab -- confira no AWS Details
}

# Se o workgroup "primary" do Athena da sua conta não tiver um "query result
# location" configurado, defina aqui um bucket/pasta pra salvar os
# resultados das queries. Se já rodou queries manualmente no console do
# Athena antes, pode deixar None -- já deve existir um configurado.
ATHENA_S3_OUTPUT = None  # ex.: "s3://<algum-bucket>/athena-results/"

In [4]:
import boto3
import awswrangler as wr
import pandas as pd

boto3_session = boto3.Session(
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"] or None,
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"] or None,
    aws_session_token=AWS_CREDENTIALS["aws_session_token"] or None,
    region_name=AWS_CREDENTIALS["region_name"],
)

# teste rápido de conexão -- se as credenciais estiverem erradas/expiradas,
# falha aqui já, antes de gastar tempo nas queries.
sts = boto3_session.client("sts")
print("Conectado como:", sts.get_caller_identity().get("Arn"))

c:\Users\Pedro\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Conectado como: arn:aws:sts::220856710497:assumed-role/voclabs/user5264977=pedropcs608@gmail.com


## 2. Databases e tabelas a comparar

In [5]:
DATABASES = {
    "v1_antigo": "state_of_data_gold",
    "v2_novo": "state_of_data_gold_v2",
}

TABELAS = [
    "q1_top_cargos",
    "q1_respondentes_por_ano",
    "q2_senioridade_salario",
    "q3_genero_por_ano",
    "q4_linguagens_por_ano",
    "q5_ia_por_ano",
    "q6_regiao",
    "q6_senioridade_regiao",
]

## 3. Buscar os dados das duas databases

Roda um `SELECT *` em cada uma das 8 tabelas, nos dois databases (16
queries no total). Se uma tabela ainda não existir num dos databases (ex.:
v2 ainda não foi catalogada), o erro fica registrado e a comparação
continua nas outras tabelas.

In [6]:
def busca_tabela(database: str, tabela: str):
    """Roda um SELECT * na tabela e devolve um DataFrame -- ou None se der
    erro (ex.: tabela ainda não criada nesse database)."""
    sql = f"SELECT * FROM {tabela}"
    try:
        return wr.athena.read_sql_query(
            sql=sql,
            database=database,
            boto3_session=boto3_session,
            s3_output=ATHENA_S3_OUTPUT,
        )
    except Exception as e:
        print(f"[ERRO] {database}.{tabela}: {e}")
        return None


dados = {}
for versao, database in DATABASES.items():
    for tabela in TABELAS:
        print(f"Buscando {database}.{tabela} ...")
        dados[(versao, tabela)] = busca_tabela(database, tabela)

print("\nConcluído.")

Buscando state_of_data_gold.q1_top_cargos ...


c:\Users\Pedro\AppData\Local\Programs\Python\Python313\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


Buscando state_of_data_gold.q1_respondentes_por_ano ...


c:\Users\Pedro\AppData\Local\Programs\Python\Python313\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


Buscando state_of_data_gold.q2_senioridade_salario ...


...

Buscando state_of_data_gold_v2.q6_regiao ...


c:\Users\Pedro\AppData\Local\Programs\Python\Python313\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


Buscando state_of_data_gold_v2.q6_senioridade_regiao ...


c:\Users\Pedro\AppData\Local\Programs\Python\Python313\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)



Concluído.


## 4. Conferência rápida — respondentes por ano

O check mais simples e mais importante: se o total de respondentes por ano
mudou entre v1 e v2, algo na ingestão/Bronze/Silver do pipeline novo
perdeu ou duplicou linhas.

In [7]:
for versao in DATABASES:
    df = dados[(versao, "q1_respondentes_por_ano")]
    print(f"--- {versao} ({DATABASES[versao]}) ---")
    if df is not None:
        print(df.sort_values("ano_pesquisa").to_string(index=False))
    else:
        print("(não foi possível buscar)")
    print()

--- v1_antigo (state_of_data_gold) ---
 ano_pesquisa  count
         2023   2822
         2024   2992
         2025   2340

--- v2_novo (state_of_data_gold_v2) ---
 ano_pesquisa  count
         2023   2822
         2024   2992
         2025   2340



## 5. Comparação tabela a tabela

Para cada uma das 8 tabelas: conta linhas dos dois lados, junta pelas
colunas de dimensão (tudo, exceto `count`) e calcula:

- **so_v1** — combinações que existiam no v1 e sumiram no v2;
- **so_v2** — combinações novas, que só aparecem no v2;
- **count_diferente** — combinações presentes nos dois lados, mas com
  `count` diferente;
- **diff_absoluta_total** — soma de `|count_v1 - count_v2|` só das
  combinações presentes nos dois lados (dá uma ideia de "quão diferente",
  além de só contar quantas linhas divergem).

In [8]:
def compara_tabela(tabela: str) -> dict:
    df_v1 = dados[("v1_antigo", tabela)]
    df_v2 = dados[("v2_novo", tabela)]

    resultado = {
        "tabela": tabela,
        "linhas_v1": None,
        "linhas_v2": None,
        "colunas_batem": None,
        "so_v1": None,
        "so_v2": None,
        "count_diferente": None,
        "diff_absoluta_total": None,
        "erro": None,
        "_merged_df": None,
    }

    if df_v1 is None or df_v2 is None:
        resultado["erro"] = "não foi possível buscar um dos dois lados (ver erro na célula da seção 3)"
        return resultado

    resultado["linhas_v1"] = len(df_v1)
    resultado["linhas_v2"] = len(df_v2)
    resultado["colunas_batem"] = set(df_v1.columns) == set(df_v2.columns)

    if not resultado["colunas_batem"]:
        resultado["erro"] = (
            f"colunas diferentes -- v1={sorted(df_v1.columns)} "
            f"v2={sorted(df_v2.columns)}"
        )
        return resultado

    dim_cols = [c for c in df_v1.columns if c != "count"]

    merged = df_v1.merge(
        df_v2, on=dim_cols, how="outer", suffixes=("_v1", "_v2"), indicator=True
    )

    resultado["so_v1"] = int((merged["_merge"] == "left_only").sum())
    resultado["so_v2"] = int((merged["_merge"] == "right_only").sum())

    ambos = merged[merged["_merge"] == "both"].copy()
    ambos["diff"] = (ambos["count_v1"] - ambos["count_v2"]).abs()
    resultado["count_diferente"] = int((ambos["diff"] > 0).sum())
    resultado["diff_absoluta_total"] = int(ambos["diff"].sum())

    resultado["_merged_df"] = merged
    return resultado


resumos = [compara_tabela(t) for t in TABELAS]
resumo_df = pd.DataFrame(resumos).drop(columns=["_merged_df"])
resumo_df

,tabela,linhas_v1,linhas_v2,colunas_batem,so_v1,so_v2,count_diferente,diff_absoluta_total,erro
0,q1_top_cargos,18,18,True,0,0,0,0,None
1,q1_respondentes_por_ano,3,3,True,0,0,0,0,None
2,q2_senioridade_salario,157,157,True,0,0,0,0,None
3,q3_genero_por_ano,12,12,True,0,0,0,0,None
4,q4_linguagens_por_ano,37,37,True,0,0,0,0,None
5,q5_ia_por_ano,15,15,True,0,0,0,0,None
6,q6_regiao,15,15,True,0,0,0,0,None
7,q6_senioridade_regiao,78,78,True,0,0,0,0,None


## 6. Resumo final — olhe aqui primeiro

In [9]:
print("Resumo por tabela:\n")
display(resumo_df)

problemas = resumo_df[
    (resumo_df["so_v1"].fillna(0) > 0)
    | (resumo_df["so_v2"].fillna(0) > 0)
    | (resumo_df["count_diferente"].fillna(0) > 0)
    | (~resumo_df["colunas_batem"].fillna(True))
    | (resumo_df["erro"].notna())
]

if problemas.empty:
    print("Nenhuma diferença encontrada -- as 8 tabelas batem entre v1 e v2.")
else:
    print("Tabelas com diferença -- investigar na seção 7:")
    display(problemas)

Resumo por tabela:



,tabela,linhas_v1,linhas_v2,colunas_batem,so_v1,so_v2,count_diferente,diff_absoluta_total,erro
0,q1_top_cargos,18,18,True,0,0,0,0,None
1,q1_respondentes_por_ano,3,3,True,0,0,0,0,None
2,q2_senioridade_salario,157,157,True,0,0,0,0,None
3,q3_genero_por_ano,12,12,True,0,0,0,0,None
4,q4_linguagens_por_ano,37,37,True,0,0,0,0,None
5,q5_ia_por_ano,15,15,True,0,0,0,0,None
6,q6_regiao,15,15,True,0,0,0,0,None
7,q6_senioridade_regiao,78,78,True,0,0,0,0,None


Nenhuma diferença encontrada -- as 8 tabelas batem entre v1 e v2.


## 7. Inspecionar uma tabela específica (opcional)

Se alguma tabela na seção 6 aparecer com diferença, troque o nome abaixo e
rode esta célula pra ver as linhas divergentes lado a lado.

In [10]:
TABELA_PARA_INSPECIONAR = "q6_senioridade_regiao"  # troque pelo nome da tabela a investigar

_r = next(r for r in resumos if r["tabela"] == TABELA_PARA_INSPECIONAR)
_merged = _r.get("_merged_df")

if _merged is None:
    print("Sem dado suficiente pra essa tabela --", _r.get("erro"))
else:
    so_um_lado = _merged[_merged["_merge"] != "both"]
    count_diferente = _merged[
        (_merged["_merge"] == "both") & (_merged["count_v1"] != _merged["count_v2"])
    ]
    print(f"Linhas só de um lado (v1 ou v2): {len(so_um_lado)}")
    display(so_um_lado)
    print(f"\nLinhas nos dois lados mas com count diferente: {len(count_diferente)}")
    display(count_diferente)

Linhas só de um lado (v1 ou v2): 0


,ano_pesquisa,regiao,senioridade,count_v1,count_v2,_merge



Linhas nos dois lados mas com count diferente: 0


,ano_pesquisa,regiao,senioridade,count_v1,count_v2,_merge
